# 17 — Production Agent Engineering

This notebook hardens the Investment AI runtime around four production controls:

```text
1. MLflow Prompt Registry
2. @staging / @production aliases
3. Unity Gateway Model Service
4. Timeout, retry, and output-token limits
```

Model Service already created manually:

```text
investment_ai.gateway.investment_ai_runtime

Primary  → claude-haiku-5-5
Fallback → claude-sonnet-5-5
```

Fallback routing stays in Unity Gateway rather than application code.


## 1. Install dependencies


In [0]:
%pip install -q --upgrade "mlflow[databricks]>=3.14.0" "openai>=2.0.0" databricks-sdk
dbutils.library.restartPython()


## 2. Configure MLflow and the governed model runtime

The application references one prompt name and one Model Service name.

The underlying prompt version and model destination can change without changing the calling code.


In [0]:
import time
import mlflow

from databricks.sdk import WorkspaceClient
from openai import OpenAI, APIConnectionError, APITimeoutError, APIStatusError

PROMPT_NAME = "investment_ai.agent.investment_ai_runtime_prompt"
MODEL_SERVICE = "investment_ai.gateway.investment_ai_runtime"

MAX_OUTPUT_TOKENS = 800
REQUEST_TIMEOUT_SECONDS = 30
MAX_ATTEMPTS = 2

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

w = WorkspaceClient()
auth_header = w.config.authenticate()["Authorization"]
databricks_token = auth_header.replace("Bearer ", "", 1)

runtime_client = OpenAI(
    api_key=databricks_token,
    base_url=f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1",
    timeout=REQUEST_TIMEOUT_SECONDS,
    max_retries=0,
)

print("Prompt:", PROMPT_NAME)
print("Runtime:", MODEL_SERVICE)
print("Max output tokens:", MAX_OUTPUT_TOKENS)


## 3. Register Prompt Version 1

Prompt versions are immutable. A new prompt change creates a new version rather than silently overwriting production behavior.


In [0]:
PROMPT_V1 = [
    {
        "role": "system",
        "content": (
            "You are an Investment AI assistant. "
            "Answer clearly and concisely using the supplied governed context. "
            "Do not invent portfolio facts."
        ),
    },
    {
        "role": "user",
        "content": (
            "Question:\n{{question}}\n\n"
            "Governed context:\n{{context}}"
        ),
    },
]

v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=PROMPT_V1,
    commit_message="Initial Investment AI production runtime prompt",
    tags={"project": "investment-ai", "purpose": "production-runtime"},
)

print("Created version:", v1.version)


## 4. Point `@production` to Version 1

Production code loads an alias rather than a fixed prompt version.


In [0]:
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production",
    version=v1.version,
)

production_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)

print("Production version:", production_prompt.version)


## 5. Register an improved Version 2

Version 2 adds stronger governance behavior and a clearer response structure.


In [0]:
PROMPT_V2 = [
    {
        "role": "system",
        "content": (
            "You are the production Investment AI assistant. "
            "Use only the supplied governed context for factual claims. "
            "Never override or recalculate a governed risk_status such as BREACH or OK. "
            "If required information is missing, say that it is unavailable. "
            "Keep the response concise. "
            "Structure the response as: Answer, Evidence, and Next step."
        ),
    },
    {
        "role": "user",
        "content": (
            "Question:\n{{question}}\n\n"
            "Governed context:\n{{context}}"
        ),
    },
]

v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=PROMPT_V2,
    commit_message="Add governance rules and concise response structure",
    tags={
        "project": "investment-ai",
        "purpose": "production-runtime",
        "change": "governance-and-structure",
    },
)

mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="staging",
    version=v2.version,
)

print("Staging version:", v2.version)


## 6. Load production and staging dynamically

At this point:

```text
@production → Version 1
@staging    → Version 2
```


In [0]:
prod_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)
staging_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@staging"
)

print("Production:", prod_prompt.version)
print("Staging   :", staging_prompt.version)


## 7. Add bounded runtime retry

Unity Gateway already handles fallback **inside** the Model Service.

Application retry is only for a transient failure returned by the Model Service itself: timeout, connection failure, HTTP 429, or HTTP 5xx.


In [0]:
def call_runtime(messages, max_attempts=MAX_ATTEMPTS):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            return runtime_client.responses.create(
                model=MODEL_SERVICE,
                input=messages,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )

        except (APITimeoutError, APIConnectionError) as e:
            last_error = e

        except APIStatusError as e:
            last_error = e
            if not (e.status_code == 429 or e.status_code >= 500):
                raise

        if attempt < max_attempts:
            sleep_seconds = 2 ** (attempt - 1)
            print(
                f"Transient runtime failure on attempt {attempt}. "
                f"Retrying in {sleep_seconds}s..."
            )
            time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Model Service failed after {max_attempts} attempts"
    ) from last_error


## 8. Use a familiar governed portfolio-risk case

Reusing an existing case keeps the focus on production engineering rather than introducing new business logic.


In [0]:
QUESTION = (
    "Does our largest holding breach the concentration policy, "
    "and what should the portfolio manager understand from the result?"
)

GOVERNED_CONTEXT = '''
Portfolio: Synthetic Growth Fund 01
Largest holding: Synthetic Compute Company 01
Position weight: 39.34%
Concentration threshold: 10.00%
Governed risk_status: BREACH
Excess weight: 29.34 percentage points
'''.strip()

print(QUESTION)
print()
print(GOVERNED_CONTEXT)


## 9. Compare production and staging prompts

The same Model Service and the same governed facts are used for both calls. Only the prompt alias changes.


In [0]:
prod_messages = prod_prompt.format(
    question=QUESTION,
    context=GOVERNED_CONTEXT,
)

staging_messages = staging_prompt.format(
    question=QUESTION,
    context=GOVERNED_CONTEXT,
)

prod_response = call_runtime(prod_messages)
staging_response = call_runtime(staging_messages)

print("===== PRODUCTION PROMPT =====")
print("Version:", prod_prompt.version)
print(prod_response.output_text)

print("\n===== STAGING PROMPT =====")
print("Version:", staging_prompt.version)
print(staging_response.output_text)


## 10. Inspect token usage

Cost-aware controls in this POC are intentionally simple:

```text
Haiku primary
+ max output-token limit
+ Sonnet only as fallback
```


In [0]:
def print_usage(label, response):
    usage = getattr(response, "usage", None)
    if usage is None:
        print(label, "- usage metadata not returned")
        return

    print(
        label,
        "| input:", getattr(usage, "input_tokens", None),
        "| output:", getattr(usage, "output_tokens", None),
        "| total:", getattr(usage, "total_tokens", None),
    )

print_usage("Production", prod_response)
print_usage("Staging   ", staging_response)


## 11. Promote the tested prompt with a rollback pointer

Before promotion, save the current production version under `@production_previous`.

Then move `@production` to the tested staging version.


In [0]:
current_production = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)

mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production_previous",
    version=current_production.version,
)

mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production",
    version=staging_prompt.version,
)

new_production = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)

print("Previous production:", current_production.version)
print("New production     :", new_production.version)


## 12. Application code stays unchanged

The app still loads `@production`; the alias now resolves to the newly promoted immutable version.


In [0]:
runtime_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)

runtime_messages = runtime_prompt.format(
    question=QUESTION,
    context=GOVERNED_CONTEXT,
)

final_response = call_runtime(runtime_messages)

print("Resolved production version:", runtime_prompt.version)
print()
print(final_response.output_text)


## 13. Model fallback remains centralized in Unity Gateway

The notebook calls only:

```text
investment_ai.gateway.investment_ai_runtime
```

Behind it:

```text
Primary
→ claude-haiku-5-5

Primary invocation fails
→ claude-sonnet-5-5
```

Fallback is an **availability policy**, not a quality-upgrade policy. Evaluation remains responsible for determining whether an answer is good enough.


# ✅ Production Agent Engineering complete

We now have:

```text
Prompt lifecycle
→ MLflow Prompt Registry

Prompt deployment
→ @staging / @production aliases

Model routing + fallback
→ Unity Gateway Model Service

Transient failures
→ bounded application retry

Latency control
→ request timeout

Cost control
→ Haiku primary + output-token limit
```

Runtime pattern:

```text
Application
    ↓
Load @production prompt
    ↓
Timeout + bounded retry
    ↓
investment_ai.gateway.investment_ai_runtime
    ↓
Unity Gateway
    ├── Haiku primary
    └── Sonnet fallback
    ↓
Response
```

### Next
**Security, Governance & Guardrails**
